# Initial Code setting

**Class Define**

In [19]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional
import time

@dataclass
class Item:
    name: str
    quantity: int
    weight: float
    rarity: str = "common"  # common / rare / epic / legendary

class InventoryNode:
    def __init__(self, item: Item):
        self.item = item
        self.prev: Optional[InventoryNode] = None
        self.next: Optional[InventoryNode] = None

class GameInventory:
    def __init__(self, max_slots: int = 20):
        self.max_slots = max_slots
        self.head: Optional[InventoryNode] = None
        self.tail: Optional[InventoryNode] = None
        self.cursor: Optional[InventoryNode] = None  # Currently selected slot (for controller)
        self.size = 0
        self._map: dict[str, InventoryNode] = {}     # Hash map for fast lookups

**Insert, Remove Item**

In [20]:
def add_item(self, item: Item, verbose: bool = True) -> bool:
    if self.size >= self.max_slots:
        if verbose: print(f"Inventory Full! Cannot add '{item.name}'")
        return False

    # Stacking logic: if the item already exists, only increase the quantity
    if item.name in self._map:
        self._map[item.name].item.quantity += item.quantity
        if verbose: print(f"Increased quantity of '{item.name}' → {self._map[item.name].item.quantity}")
        return True

    node = InventoryNode(item)
    self._map[item.name] = node

    if not self.head:
        self.head = self.tail = node
        self.cursor = node
    else:
        node.prev = self.tail
        self.tail.next = node
        self.tail = node

    self.size += 1
    if verbose: print(f"Acquired: [{item.rarity.upper()}] {item.name} x{item.quantity}")
    return True

def remove_item(self, name: str, quantity: int = 1, verbose: bool = True) -> bool:
    if name not in self._map:
        if verbose: print(f"'{name}' is not in your inventory.")
        return False

    node = self._map[name]
    node.item.quantity -= quantity

    if node.item.quantity <= 0:
        # Move the cursor away if it points to the node being deleted
        if self.cursor is node:
            self.cursor = node.next or node.prev

        self._remove_node(node)
        del self._map[name]
        self.size -= 1
        if verbose: print(f"❌ '{name}' depleted. Slot completely removed.")
    else:
        if verbose: print(f"📉 Used '{name}' → Remaining: {node.item.quantity}")
    return True

# Registering functions as methods of GameInventory
GameInventory.add_item = add_item
GameInventory.remove_item = remove_item

**Controller Search**

In [21]:
def navigate_next(self) -> Optional[Item]:
    if self.cursor and self.cursor.next:
        self.cursor = self.cursor.next
        print(f" Moved ▶ Current Selection: {self.cursor.item.name}")
        return self.cursor.item
    print(" Movement Failed: No more items to the right.")
    return None

def navigate_prev(self) -> Optional[Item]:
    if self.cursor and self.cursor.prev:
        self.cursor = self.cursor.prev
        print(f" Moved ◀ Current Selection: {self.cursor.item.name}")
        return self.cursor.item
    print(" Movement Failed: No more items to the left.")
    return None

GameInventory.navigate_next = navigate_next
GameInventory.navigate_prev = navigate_prev

# Demo code

**Code Display**

basic operations we explained

In [22]:
print("=== Inventory Structure & Controller Navigation Demo ===")
inv = GameInventory(max_slots=5)

inv.add_item(Item("Red_Potion", quantity=2, weight=0.5))
inv.add_item(Item("Longsword", quantity=1, weight=3.0, rarity="rare"))
inv.add_item(Item("Iron_Shield", quantity=1, weight=5.0, rarity="rare"))
inv.add_item(Item("Red_Potion", quantity=3, weight=0.5)) # Checking item stacking

print(f"\nInitial Cursor Position: {inv.cursor.item.name if inv.cursor else 'None'}")
inv.navigate_next()  # Move cursor to Longsword
inv.navigate_next()  # Move cursor to Iron_Shield
inv.navigate_prev()  # Move cursor back to Longsword

=== Inventory Structure & Controller Navigation Demo ===
Acquired: [COMMON] Red_Potion x2
Acquired: [RARE] Longsword x1
Acquired: [RARE] Iron_Shield x1
Increased quantity of 'Red_Potion' → 5

Initial Cursor Position: Red_Potion
 Moved ▶ Current Selection: Longsword
 Moved ▶ Current Selection: Iron_Shield
 Moved ◀ Current Selection: Longsword


Item(name='Longsword', quantity=1, weight=3.0, rarity='rare')

**DLL search limitation**

find the last item of inventory

**Search without / with hash map**

In [23]:
def search_without_hash(self, name: str) -> Optional[Item]:
    """[O(N)] Sequential search traversing from Head to Tail"""
    current = self.head
    while current:
        if current.item.name == name:
            return current.item
        current = current.next
    return None

def search_with_hash(self, name: str) -> Optional[Item]:
    """[O(1)] Instant lookup via Hash Map address"""
    if name in self._map:
        return self._map[name].item
    return None

GameInventory.search_without_hash = search_without_hash
GameInventory.search_with_hash = search_with_hash

In [24]:
print("=== 10,000 Items Mass Injection & Search Speed Comparison ===")
# Create a larger inventory to hold 10,000 items
test_inv = GameInventory(max_slots=15000)

for i in range(10000):
    # Set verbose=False to prevent Colab output from freezing
    test_inv.add_item(Item(name=f"Item_{i}", quantity=1, weight=0.1), verbose=False)
print(f"Successfully injected {test_inv.size} items into the inventory!\n")

target_item = "Item_9999"

# without Hash Map
start_time = time.perf_counter()
test_inv.search_without_hash(target_item)
time_without_hash = (time.perf_counter() - start_time) * 1000
print(f"🐌 [O(N) Linear Search] Elapsed Time: {time_without_hash:.4f} ms")

# with Hash Map
start_time = time.perf_counter()
test_inv.search_with_hash(target_item)
time_with_hash = (time.perf_counter() - start_time) * 1000
print(f"⚡ [O(1) Hash Map Search]   Elapsed Time: {time_with_hash:.4f} ms")

print(f"\n💡 Conclusion: Combining a Hash Map makes lookup approx. {time_without_hash / (time_with_hash + 1e-9):.1f} times faster!")

=== 10,000 Items Mass Injection & Search Speed Comparison ===
Successfully injected 10000 items into the inventory!

🐌 [O(N) Linear Search] Elapsed Time: 0.7860 ms
⚡ [O(1) Hash Map Search]   Elapsed Time: 0.0705 ms

💡 Conclusion: Combining a Hash Map makes lookup approx. 11.2 times faster!


**My inventory sorting**

make your most recent used item in the front of the inventory

**LRU Cache**


In [25]:
def _remove_node(self, node: InventoryNode):
    """Disconnect links around the node from the middle of the DLL"""
    if node.prev: node.prev.next = node.next
    else: self.head = node.next

    if node.next: node.next.prev = node.prev
    else: self.tail = node.prev

def _insert_at_head(self, node: InventoryNode):
    """Prepend the node to the very front (Head) of the DLL"""
    node.next = self.head
    node.prev = None
    if self.head: self.head.prev = node
    self.head = node
    if not self.tail: self.tail = node

def use_item(self, name: str) -> bool:
    if name not in self._map:
        print(f"'{name}' not found.")
        return False

    node = self._map[name]

    # [6min Interactive Workshop - Code Blank Section]
    # --------------------------------------------------
    self._remove_node(node)      # 1. Remove from current position
    self._insert_at_head(node)   # 2. Re-insert at the Head
    # --------------------------------------------------

    print(f"🔥 Successfully used '{name}'! Moved to the front of the inventory.")
    return True

GameInventory._remove_node = _remove_node
GameInventory._insert_at_head = _insert_at_head
GameInventory.use_item = use_item

In [26]:
print("=== Verifying Recently Used Item Sorting (LRU) ===")
lru_inv = GameInventory(max_slots=10)

# Build 4 sequential items
for i in range(4):
    lru_inv.add_item(Item(name=f"Item_{i}", quantity=1, weight=1.0), verbose=False)

print("■ Inventory layout BEFORE using an item:")
print(f"  - Front (Head): {lru_inv.head.item.name}")
print(f"  - Back (Tail):  {lru_inv.tail.item.name}\n")

# Using 'Item_1' which is nested in the middle
lru_inv.use_item("Item_1")

print("\n■ Inventory layout AFTER using the item:")
print(f"  - Front (Head): {lru_inv.head.item.name} ◀ [Success] Successfully shifted to Head!")
print(f"  - Next to Head: {lru_inv.head.next.item.name}")
print(f"  - Back (Tail):  {lru_inv.tail.item.name}")

=== Verifying Recently Used Item Sorting (LRU) ===
■ Inventory layout BEFORE using an item:
  - Front (Head): Item_0
  - Back (Tail):  Item_3

🔥 Successfully used 'Item_1'! Moved to the front of the inventory.

■ Inventory layout AFTER using the item:
  - Front (Head): Item_1 ◀ [Success] Successfully shifted to Head!
  - Next to Head: Item_0
  - Back (Tail):  Item_3
